# 2.3 Обучение binary-моделей POS

In [ ]:
# PROJECT_ROOT — ветка dynamic, в том числе при запуске Jupyter из общего каталога.
from pathlib import Path
import sys, os
_branch = 'dynamic'
_cwd = Path.cwd().resolve()
_candidates = [_cwd, _cwd / _branch, _cwd.parent / _branch]
PROJECT_ROOT = next((x for x in _candidates if x.name == _branch and (x / 'srcc' / 'nbc_project').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f'Не найдена папка {_branch}/srcc/nbc_project; запустите из ветки или её родителя')
SRC_PATH = PROJECT_ROOT / 'srcc'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT:', PROJECT_ROOT)
EXPERIMENT_NAME = '202610_dynamic'
RUN_TAG = '202610_dynamic'


In [ ]:
import json
from pathlib import Path
import pandas as pd

from nbc_project.training_workflow import (
    TrainJobConfig,
    run_jobs,
    summarize_results,
    run_optional_rolling,
    run_optional_visuals,
)
from nbc_project.feature_lists import get_feature_list


with open(PROJECT_ROOT / 'configs' / 'training_jobs.json', encoding='utf-8') as f:
    TRAINING_CONFIG = json.load(f)

DEFAULT_SPLIT = TRAINING_CONFIG['default_split']
DEFAULT_SPLIT

## Конфигурация обучения

In [ ]:
configs = [
    TrainJobConfig(**{**DEFAULT_SPLIT, **cfg})
    for cfg in TRAINING_CONFIG['pos_binary']
]

for cfg in configs:
    print(cfg)
# Split/output конкретного эксперимента
for cfg in configs:
    cfg.train_end_exclusive='2026-10-01'
    cfg.valid_month='2026-10-01'
    cfg.test_month=None
    cfg.split_suffix='202610_dynamic'
    cfg.legacy_test_suffix=None
    cfg.output_split_dir=str(PROJECT_ROOT / 'experiment_outputs' / '202610_dynamic' / 'splits' / 'binary_bin')
    cfg.model_output_dir=str(PROJECT_ROOT / 'experiment_outputs' / '202610_dynamic' / 'models' / 'binary_bin')
configs

for cfg in configs:
    segment = 'T' if cfg.model_key.endswith('_T') else 'C'
    cfg.model_filename = f"model_{segment}_202610_dynamic.cbm"
configs


### Опционально: dynamic feature selection — v6

- `USE_DYNAMIC_FS = False`: **current** остаётся на старом static feature set (`acrm.common_segmentation`).
- `USE_DYNAMIC_FS = True`: модель читает отдельный **full modular** dataset (`feature_store_inc_modular`) и запускает выбранный dynamic FS.
  
Dynamic FS сначала работает на sample, а затем полный parquet читается только по выбранным колонкам.


In [ ]:
USE_DYNAMIC_FS = True

# Финальная конфигурация после benchmark и переноса выбранных подходов
# на модели, для которых отдельный benchmark не прогоняли.
FS_BY_MODEL = {'binary_T': 'psi_importance_auto_n', 'binary_C': 'psi_importance_auto_n'}

DYNAMIC_FS_PRESETS = {
    "psi_importance_auto_n": {
        "method": "psi_importance",
        "psi_threshold": 0.20,
        "psi_reference_months": 3,
        "auto_top_n": True,
        "top_n_candidates": [100, 150, 200, 250, 300],
        "sample_rows": 150_000,
        "selector_iterations": 200,
        "report_dir": "reports/feature_selection",
    },
    "psi_rfe_loss_auto_n": {
        "method": "psi_rfe",
        "psi_threshold": 0.20,
        "psi_reference_months": 3,
        "rfe_algorithm": "loss_function_change",
        "rfe_steps": 5,
        "auto_top_n": True,
        "top_n_candidates": [100, 150, 200, 250, 300],
        "sample_rows": 150_000,
        "selector_iterations": 180,
        "report_dir": "reports/feature_selection",
    },
}

CORRELATION_PREFILTER = {
    "drop_correlated_features": True,
    "correlation_threshold": 0.95,
    "correlation_method": "pearson",
    "correlation_max_rows": 20_000,
    "correlation_min_unique_values": 20,
}

for _cfg in DYNAMIC_FS_PRESETS.values():
    _cfg.update(CORRELATION_PREFILTER)

PROTECTED_OFFER_FEATURES = [
    c
    for c in get_feature_list("data_fm_must_have_features")
    if c.startswith("offered_percent_cat_")
]

if not PROTECTED_OFFER_FEATURES:
    PROTECTED_OFFER_FEATURES = [
        c
        for c in get_feature_list("must_have_offered_percent_features")
        if c.startswith("offered_percent_cat_")
    ]

def make_fs_config(model_key):
    preset = FS_BY_MODEL[model_key]
    cfg = DYNAMIC_FS_PRESETS[preset].copy()
    cfg["protected_features"] = PROTECTED_OFFER_FEATURES.copy()
    return cfg

print("FINAL FS_BY_MODEL:", FS_BY_MODEL)
print("protected offered_percent:", len(PROTECTED_OFFER_FEATURES))


FULL_DATA_BY_SEGMENT = {
    "T": str(PROJECT_ROOT / "data" / "datasets" / "oct_202610" / "mae_nbc_fm_loyalty_dataset_modular_full"),
    "C": str(PROJECT_ROOT / "data" / "datasets" / "oct_202610" / "mae_nbc_fm_noloyalty_dataset_modular_full"),
}

for cfg in configs:
    segment = "T" if cfg.model_key.endswith("_T") else "C"
    preset = FS_BY_MODEL[cfg.model_key]

    cfg.input_path = FULL_DATA_BY_SEGMENT[segment]
    cfg.ignore_cols = list(
        dict.fromkeys(cfg.ignore_cols + ["period_dt", "n_treat", "rn"])
    )
    cfg.feature_selection = make_fs_config(cfg.model_key)

    cfg.model_filename = cfg.model_filename.replace(
        ".cbm",
        f"_{preset}.cbm",
    )

print("USE_DYNAMIC_FS:", USE_DYNAMIC_FS)
for cfg in configs:
    print(
        cfg.model_key,
        "->",
        FS_BY_MODEL[cfg.model_key],
        "->",
        cfg.model_filename,
    )

configs


In [ ]:
# HDFS: ровно одна октябрьская выгрузка под отдельным месячным каталогом.
import subprocess, shutil
OCT_DATA = PROJECT_ROOT / 'data' / 'datasets' / 'oct_202610'
OCT_DATA.mkdir(parents=True, exist_ok=True)
for table in ['mae_nbc_fm_loyalty_dataset_modular_full', 'mae_nbc_fm_noloyalty_dataset_modular_full']:
    target = OCT_DATA / table
    if not target.exists():
        paths = [f'/warehouse/tablespace/{kind}/hive/sandbox_crft.db/{table}' for kind in ('managed','external')]
        source = next((p for p in paths if subprocess.run(['hdfs','dfs','-test','-d',p]).returncode == 0), None)
        if source is None:
            raise FileNotFoundError(f'Октябрьская витрина {table} недоступна в HDFS: {paths}')
        subprocess.run(['hdfs','dfs','-copyToLocal',source,str(OCT_DATA)],check=True)
    if not any(target.rglob('*.parq*')):
        raise RuntimeError(f'Нет Parquet в {target}')
print('Dynamic full marts готовы:', OCT_DATA)

# здесь меняем iterations
FAST_TEST_MODE = False

if FAST_TEST_MODE:
    for cfg in configs:
        cfg.iterations = 30
        cfg.learning_rate = 0.08
        cfg.depth = 4
        cfg.early_stopping_rounds = 30

configs

## Обучение T/C-сегментов

In [ ]:
results = run_jobs(configs)

## Сводка метрик

In [ ]:
summary = summarize_results(results)
summary

## 4. Визуализации по таргетам, моделям, метрикам и feature importance

Блок строит презентационные графики для каждой модели/сегмента: target distribution, boxplot таргета, metrics bar, диагностику prediction/fact или ROC/PR/calibration, confusion matrix для binary и feature importance.

In [ ]:
RUN_VISUALS = True

if RUN_VISUALS:
    visuals = run_optional_visuals(
        results,
        output_dir='reports/training_visuals',
    )
    visuals


## 5. Rolling train / walk-forward backtest

In [ ]:
RUN_ROLLING_BACKTEST = True

ROLLING_MODEL_PARAMS = {
    'iterations': 300,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 5,
    'random_seed': 42,
    'early_stopping_rounds': 50,
    'verbose': 100,
}

if RUN_ROLLING_BACKTEST:
    rolling = run_optional_rolling(
        results,
        output_dir='reports/rolling_backtest',
        min_train_months=3,
        model_params=ROLLING_MODEL_PARAMS,
    )
    rolling


## 6. Итоговые артефакты

После Run All будут сохранены:

- модели и metadata в директории из `model_output_dir`;
- split-файлы train/valid/test;
- `*_feature_importance.csv`;
- графики в `reports/training_visuals/<model_key>/`;
- rolling metrics/heatmap в `reports/rolling_backtest/<model_key>/`.

In [ ]:
{
    'summary': summary,
    'visuals_dir': 'reports/training_visuals',
    'rolling_dir': 'reports/rolling_backtest' if RUN_ROLLING_BACKTEST else None,
}
